In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/ml-20m")

ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")
tags = pd.read_csv(DATA_DIR / "tags.csv")
genome_tags = pd.read_csv(DATA_DIR / "genome-tags.csv")
genome_scores = pd.read_csv(DATA_DIR / "genome-scores.csv")
links = pd.read_csv(DATA_DIR / "links.csv")

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s", errors="coerce")
tags["timestamp"] = pd.to_datetime(tags["timestamp"], unit="s", errors="coerce")

movies["year"] = movies["title"].str.extract(r"\((\d{4})\)").astype(float)
movies["clean_title"] = movies["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)
movies["genres_list"] = movies["genres"].fillna("(no genres listed)").str.split("|")

tags = tags.dropna(subset=["tag"]).copy()
tags["tag"] = tags["tag"].astype(str).str.strip()
tags = tags[tags["tag"] != ""].copy()

print("ratings shape:", ratings.shape)
print("movies shape:", movies.shape)
print("tags shape:", tags.shape)
print("genome_tags shape:", genome_tags.shape)
print("genome_scores shape:", genome_scores.shape)
print("links shape:", links.shape)

print("\nratings time range:", ratings["timestamp"].min(), "->", ratings["timestamp"].max())
print("tags time range   :", tags["timestamp"].min(), "->", tags["timestamp"].max())

print("\nnulls")
print("ratings:\n", ratings[["userId", "movieId", "rating", "timestamp"]].isna().sum())
print("movies:\n", movies[["movieId", "title", "genres", "year"]].isna().sum())
print("tags:\n", tags[["userId", "movieId", "tag", "timestamp"]].isna().sum())

ratings shape: (20000263, 4)
movies shape: (27278, 6)
tags shape: (465541, 4)
genome_tags shape: (1128, 2)
genome_scores shape: (11709768, 3)
links shape: (27278, 3)

ratings time range: 1995-01-09 11:46:44 -> 2015-03-31 06:40:02
tags time range   : 2005-12-24 13:00:10 -> 2015-03-31 03:09:12

nulls
ratings:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
movies:
 movieId     0
title       0
genres      0
year       22
dtype: int64
tags:
 userId       0
movieId      0
tag          0
timestamp    0
dtype: int64


In [2]:

pair_counts = ratings.groupby(["userId", "movieId"]).size()

print("total rating rows:", len(ratings))
print("unique user-movie pairs:", len(pair_counts))
print("pairs with repeats:", (pair_counts > 1).sum())
print("rows inside repeated pairs:", pair_counts[pair_counts > 1].sum())
print("max repeats for one pair:", pair_counts.max())

ratings_last = (
    ratings
    .sort_values(["userId", "movieId", "timestamp"])
    .drop_duplicates(["userId", "movieId"], keep="last")
    .reset_index(drop=True)
)

print("\nafter keeping only the last rating per user-movie")
print("rows:", len(ratings_last))

for thr in [0.5, 3.5, 4.0]:
    x = ratings_last[ratings_last["rating"] >= thr].copy()
    user_len = x.groupby("userId").size()
    item_len = x.groupby("movieId").size()

    print(f"\nratings >= {thr}")
    print("rows:", len(x))
    print("users:", x["userId"].nunique())
    print("items:", x["movieId"].nunique())
    print("avg interactions per user:", round(user_len.mean(), 2))
    print("median interactions per user:", round(user_len.median(), 2))
    print("avg interactions per item:", round(item_len.mean(), 2))
    print("median interactions per item:", round(item_len.median(), 2))

total rating rows: 20000263
unique user-movie pairs: 20000263
pairs with repeats: 0
rows inside repeated pairs: 0
max repeats for one pair: 1

after keeping only the last rating per user-movie
rows: 20000263

ratings >= 0.5
rows: 20000263
users: 138493
items: 26744
avg interactions per user: 144.41
median interactions per user: 68.0
avg interactions per item: 747.84
median interactions per item: 18.0

ratings >= 3.5
rows: 12195566
users: 138362
items: 22884
avg interactions per user: 88.14
median interactions per user: 42.0
avg interactions per item: 532.93
median interactions per item: 16.0

ratings >= 4.0
rows: 9995410
users: 138287
items: 20720
avg interactions per user: 72.28
median interactions per user: 37.0
avg interactions per item: 482.4
median interactions per item: 15.0


In [3]:


def show_seq_stats(df, name):
    seq_len = (
        df.sort_values(["userId", "timestamp"])
          .groupby("userId")
          .size()
          .sort_values()
    )

    print(f"\n{name}")
    print("users:", len(seq_len))
    print(seq_len.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

    for L in [5, 10, 20, 50, 100, 200]:
        pct = (seq_len >= L).mean() * 100
        print(f"users with seq >= {L}: {pct:.2f}%")

for thr in [0.5, 3.5, 4.0]:
    x = ratings_last[ratings_last["rating"] >= thr].copy()
    show_seq_stats(x, f"sequence stats for ratings >= {thr}")


sequence stats for ratings >= 0.5
users: 138493
0.25      35.00
0.50      68.00
0.75     155.00
0.90     334.00
0.95     520.00
0.99    1113.08
dtype: float64
users with seq >= 5: 100.00%
users with seq >= 10: 100.00%
users with seq >= 20: 100.00%
users with seq >= 50: 61.60%
users with seq >= 100: 37.98%
users with seq >= 200: 19.37%

sequence stats for ratings >= 3.5
users: 138362
0.25     21.0
0.50     42.0
0.75    100.0
0.90    211.0
0.95    316.0
0.99    636.0
dtype: float64
users with seq >= 5: 99.36%
users with seq >= 10: 95.95%
users with seq >= 20: 77.33%
users with seq >= 50: 44.75%
users with seq >= 100: 25.04%
users with seq >= 200: 10.81%

sequence stats for ratings >= 4.0
users: 138287
0.25     19.0
0.50     37.0
0.75     84.0
0.90    170.0
0.95    249.0
0.99    492.0
dtype: float64
users with seq >= 5: 98.84%
users with seq >= 10: 93.86%
users with seq >= 20: 73.58%
users with seq >= 50: 40.50%
users with seq >= 100: 20.71%
users with seq >= 200: 7.62%


In [4]:


tag_stats = (
    tags.groupby("movieId")
        .agg(
            raw_tag_rows=("tag", "size"),
            raw_unique_tags=("tag", "nunique")
        )
        .reset_index()
)

genome_stats = (
    genome_scores.groupby("movieId")
                 .agg(
                     genome_dims=("tagId", "size"),
                     genome_mean_rel=("relevance", "mean")
                 )
                 .reset_index()
)

movie_sem = (
    movies[["movieId", "clean_title", "genres", "year"]]
    .merge(tag_stats, on="movieId", how="left")
    .merge(genome_stats, on="movieId", how="left")
)

movie_sem["has_raw_tags"] = movie_sem["raw_tag_rows"].fillna(0) > 0
movie_sem["has_genome"] = movie_sem["genome_dims"].fillna(0) > 0
movie_sem["has_both"] = movie_sem["has_raw_tags"] & movie_sem["has_genome"]

print("all movies")
print("total movies:", len(movie_sem))
print("movies with raw tags:", int(movie_sem["has_raw_tags"].sum()))
print("movies with genome:", int(movie_sem["has_genome"].sum()))
print("movies with both:", int(movie_sem["has_both"].sum()))
print("mean raw tag rows per movie:", round(movie_sem["raw_tag_rows"].fillna(0).mean(), 2))
print("mean unique raw tags per movie:", round(movie_sem["raw_unique_tags"].fillna(0).mean(), 2))

covered_genome = movie_sem[movie_sem["has_genome"]]
if len(covered_genome) > 0:
    print("genome dims min/max:", int(covered_genome["genome_dims"].min()), int(covered_genome["genome_dims"].max()))

active_movie_ids = ratings_last["movieId"].unique()
active_sem = movie_sem[movie_sem["movieId"].isin(active_movie_ids)].copy()

print("\nactive movies only")
print("active movies:", len(active_sem))
print("active movies with raw tags:", int(active_sem["has_raw_tags"].sum()))
print("active movies with genome:", int(active_sem["has_genome"].sum()))
print("active movies with both:", int(active_sem["has_both"].sum()))

print("\nmost common raw tags")
print(tags["tag"].str.lower().value_counts().head(30))

all movies
total movies: 27278
movies with raw tags: 19545
movies with genome: 10381
movies with both: 9827
mean raw tag rows per movie: 17.07
mean unique raw tags per movie: 7.35
genome dims min/max: 1128 1128

active movies only
active movies: 26744
active movies with raw tags: 19011
active movies with genome: 10370
active movies with both: 9816

most common raw tags
tag
sci-fi                3576
based on a book       3307
atmospheric           3169
comedy                3078
action                3068
nudity (topless)      2646
surreal               2528
twist ending          2367
bd-r                  2334
funny                 2253
quirky                2034
dystopia              2015
classic               1971
stylized              1944
dark comedy           1910
romance               1874
fantasy               1850
psychology            1763
time travel           1572
disturbing            1524
visually appealing    1511
aliens                1439
social commentary     1424
tho

In [5]:


def split_preview(df, name, min_hist=5):
    df = df.sort_values(["userId", "timestamp"]).copy()

    user_counts = df.groupby("userId").size()
    keep_users = user_counts[user_counts >= min_hist].index
    df = df[df["userId"].isin(keep_users)].copy()

    df["pos"] = df.groupby("userId").cumcount()
    df["n"] = df.groupby("userId")["movieId"].transform("size")

    train = df[df["pos"] < df["n"] - 2].copy()
    val = df[df["pos"] == df["n"] - 2].copy()
    test = df[df["pos"] == df["n"] - 1].copy()

    train_items = set(train["movieId"].unique())
    val_unseen = (~val["movieId"].isin(train_items)).mean() * 100
    test_unseen = (~test["movieId"].isin(train_items)).mean() * 100

    print(f"\n{name}")
    print("users kept:", df["userId"].nunique())
    print("train rows:", len(train))
    print("val rows:", len(val))
    print("test rows:", len(test))
    print("train items:", train["movieId"].nunique())
    print("val unseen item %:", round(val_unseen, 2))
    print("test unseen item %:", round(test_unseen, 2))

for thr in [3.5, 4.0]:
    x = ratings_last[ratings_last["rating"] >= thr].copy()
    split_preview(x, f"chrono split preview for ratings >= {thr}", min_hist=5)


chrono split preview for ratings >= 3.5
users kept: 137477
train rows: 11917990
val rows: 137477
test rows: 137477
train items: 22768
val unseen item %: 0.04
test unseen item %: 0.04

chrono split preview for ratings >= 4.0
users kept: 136677
train rows: 9717328
val rows: 136677
test rows: 136677
train items: 20575
val unseen item %: 0.04
test unseen item %: 0.07


In [6]:

#final data check for preprocessing
x = ratings[ratings["rating"] >= 3.5].copy()

print("rows:", len(x))
print("users:", x["userId"].nunique())
print("items:", x["movieId"].nunique())

user_counts = x.groupby("userId").size()
item_counts = x.groupby("movieId").size()

print("\nuser count quantiles")
print(user_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nitem count quantiles")
print(item_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nusers with at least 5 interactions:", (user_counts >= 5).sum())
print("items with at least 5 interactions:", (item_counts >= 5).sum())
print("items with at least 10 interactions:", (item_counts >= 10).sum())

rows: 12195566
users: 138362
items: 22884

user count quantiles
0.25     21.0
0.50     42.0
0.75    100.0
0.90    211.0
0.95    316.0
0.99    636.0
dtype: float64

item count quantiles
0.25        3.00
0.50       16.00
0.75      142.00
0.90      869.00
0.95     2476.85
0.99    10569.70
dtype: float64

users with at least 5 interactions: 137477
items with at least 5 interactions: 15405
items with at least 10 interactions: 12988


In [7]:


#top tags per movie preview
tag_counts = (
    tags.assign(tag_lower=tags["tag"].str.lower())
        .groupby(["movieId", "tag_lower"])
        .size()
        .reset_index(name="count")
        .sort_values(["movieId", "count", "tag_lower"], ascending=[True, False, True])
)

top_tags_preview = tag_counts.groupby("movieId").head(5).copy()

sample_movies = [1, 50, 260, 296, 318]
print(
    top_tags_preview[top_tags_preview["movieId"].isin(sample_movies)]
    .sort_values(["movieId", "count"], ascending=[True, False])
    .to_string(index=False)
)

 movieId           tag_lower  count
       1               pixar     79
       1           animation     49
       1              disney     28
       1           tom hanks     23
       1  computer animation     20
      50        twist ending    139
      50        kevin spacey     89
      50            suspense     58
      50         complicated     57
      50     organized crime     44
     260              sci-fi     86
     260               space     64
     260       harrison ford     49
     260             fantasy     43
     260           adventure     36
     296   quentin tarantino    217
     296         dark comedy    117
     296           nonlinear    113
     296   samuel l. jackson    102
     296 multiple storylines     93
     318      morgan freeman    110
     318              prison    104
     318        twist ending     90
     318        stephen king     85
     318   thought-provoking     75


In [8]:

#year bucket preview
movies["year_bucket"] = pd.cut(
    movies["year"],
    bins=[1900, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020],
    include_lowest=True
)

print(movies["year_bucket"].value_counts(dropna=False).sort_index())


year_bucket
(1899.999, 1950.0]    2542
(1950.0, 1960.0]      1395
(1960.0, 1970.0]      1717
(1970.0, 1980.0]      2059
(1980.0, 1990.0]      2725
(1990.0, 2000.0]      4671
(2000.0, 2010.0]      8224
(2010.0, 2020.0]      3909
NaN                     36
Name: count, dtype: int64


In [9]:

# filtered semantic coverage
filtered_movie_ids = set(x["movieId"].unique())

movie_sem_small = movies[["movieId", "clean_title", "genres", "year", "year_bucket"]].copy()
movie_sem_small["has_raw_tags"] = movie_sem_small["movieId"].isin(tags["movieId"].unique())
movie_sem_small["has_genome"] = movie_sem_small["movieId"].isin(genome_scores["movieId"].unique())

movie_sem_small = movie_sem_small[movie_sem_small["movieId"].isin(filtered_movie_ids)].copy()
movie_sem_small["has_both"] = movie_sem_small["has_raw_tags"] & movie_sem_small["has_genome"]

print("filtered items:", len(movie_sem_small))
print("with raw tags:", int(movie_sem_small["has_raw_tags"].sum()))
print("with genome:", int(movie_sem_small["has_genome"].sum()))
print("with both:", int(movie_sem_small["has_both"].sum()))

filtered items: 22884
with raw tags: 17791
with genome: 10362
with both: 9810


____
# 3

In [10]:

# make the final filtered interaction table
x = ratings[ratings["rating"] >= 3.5].copy()

def kcore_filter(df, user_min=5, item_min=5):
    df = df.copy()

    while True:
        n_before = len(df)

        good_users = df["userId"].value_counts()
        good_users = good_users[good_users >= user_min].index
        df = df[df["userId"].isin(good_users)].copy()

        good_items = df["movieId"].value_counts()
        good_items = good_items[good_items >= item_min].index
        df = df[df["movieId"].isin(good_items)].copy()

        if len(df) == n_before:
            break

    return df

interactions = kcore_filter(x, user_min=5, item_min=5)
interactions = interactions.sort_values(["userId", "timestamp"]).reset_index(drop=True)

print("final filtered interactions")
print("rows:", len(interactions))
print("users:", interactions["userId"].nunique())
print("items:", interactions["movieId"].nunique())

user_counts = interactions.groupby("userId").size()
item_counts = interactions.groupby("movieId").size()

print("\nuser count quantiles")
print(user_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nitem count quantiles")
print(item_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

final filtered interactions
rows: 12178563
users: 137474
items: 15405

user count quantiles
0.25     21.0
0.50     43.0
0.75    100.0
0.90    211.0
0.95    317.0
0.99    636.0
dtype: float64

item count quantiles
0.25       15.00
0.50       64.00
0.75      332.00
0.90     1620.60
0.95     3903.60
0.99    13399.08
dtype: float64


In [11]:

# map userId and movieId into model ids
user_ids = np.sort(interactions["userId"].unique())
movie_ids = np.sort(interactions["movieId"].unique())

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {m: i + 1 for i, m in enumerate(movie_ids)}   # 0 is kept for padding
idx2item = {i: m for m, i in item2idx.items()}

interactions["user_idx"] = interactions["userId"].map(user2idx)
interactions["item_idx"] = interactions["movieId"].map(item2idx)

print("num users:", len(user2idx))
print("num items:", len(item2idx))
print("\ninteractions preview")
print(
    interactions[["userId", "movieId", "user_idx", "item_idx", "rating", "timestamp"]]
    .head(10)
    .to_string(index=False)
)

num users: 137474
num items: 15405

interactions preview
 userId  movieId  user_idx  item_idx  rating           timestamp
      1      924         0       885     3.5 2004-09-10 03:06:38
      1      919         0       880     3.5 2004-09-10 03:07:01
      1     2683         0      2530     3.5 2004-09-10 03:07:30
      1     1584         0      1500     3.5 2004-09-10 03:07:36
      1     1079         0      1034     4.0 2004-09-10 03:07:45
      1     2959         0      2801     4.0 2004-09-10 03:08:18
      1      337         0       331     3.5 2004-09-10 03:08:29
      1     3996         0      3780     4.0 2004-09-10 03:08:47
      1      151         0       147     4.0 2004-09-10 03:08:54
      1      112         0       110     3.5 2004-09-10 03:09:00


In [12]:

#build train / val / test sequences for SASRec
user_seq = (
    interactions.sort_values(["user_idx", "timestamp"])
    .groupby("user_idx")["item_idx"]
    .apply(list)
)

split_rows = []

for user_idx, seq in user_seq.items():
    split_rows.append({
        "user_idx": user_idx,
        "train_seq": seq[:-2],
        "val_seq": seq[:-2],
        "val_target": seq[-2],
        "test_seq": seq[:-1],
        "test_target": seq[-1],
        "full_len": len(seq)
    })

splits = pd.DataFrame(split_rows)

print("number of users in splits:", len(splits))
print("train seq length quantiles")
print(splits["train_seq"].apply(len).quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nsplit preview")
print(splits.head(5).to_string(index=False))


number of users in splits: 137474
train seq length quantiles
0.25     19.0
0.50     41.0
0.75     98.0
0.90    209.0
0.95    315.0
0.99    634.0
Name: train_seq, dtype: float64

split preview
 user_idx                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                train_seq                                                                                                                                                                                                                      

In [13]:


# build item metadata table
year_labels = [
    "<=1950",
    "1951-1960",
    "1961-1970",
    "1971-1980",
    "1981-1990",
    "1991-2000",
    "2001-2010",
    "2011-2020"
]

movies["year_bucket"] = pd.cut(
    movies["year"],
    bins=[0, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020],
    labels=year_labels,
    include_lowest=True
)

movies["year_bucket"] = movies["year_bucket"].astype("object")
movies["year_bucket"] = movies["year_bucket"].fillna("unknown")

tag_counts = (
    tags.assign(tag_lower=tags["tag"].str.lower())
    .groupby(["movieId", "tag_lower"])
    .size()
    .reset_index(name="count")
    .sort_values(["movieId", "count", "tag_lower"], ascending=[True, False, True])
)

top_tags = tag_counts.groupby("movieId").head(5).copy()

top_tags_text = (
    top_tags.groupby("movieId")["tag_lower"]
    .apply(lambda s: " | ".join(s.tolist()))
    .reset_index(name="top_tags_text")
)

item_meta = movies[movies["movieId"].isin(movie_ids)].copy()
item_meta = item_meta.merge(top_tags_text, on="movieId", how="left")
item_meta["top_tags_text"] = item_meta["top_tags_text"].fillna("")

item_meta["genres_text"] = item_meta["genres"].fillna("").str.replace("|", " ", regex=False)

item_meta["movie_text"] = (
    item_meta["clean_title"].fillna("") + ". " +
    item_meta["genres_text"].fillna("") + ". " +
    item_meta["top_tags_text"].fillna("")
).str.strip()

genre_dummies = item_meta["genres"].str.get_dummies(sep="|")
item_meta = pd.concat([item_meta, genre_dummies], axis=1)

item_meta["item_idx"] = item_meta["movieId"].map(item2idx)
item_meta = item_meta.sort_values("item_idx").reset_index(drop=True)

print("item_meta shape:", item_meta.shape)
print("\nitem_meta preview")
print(
    item_meta[["item_idx", "movieId", "clean_title", "year_bucket", "top_tags_text", "movie_text"]]
    .head(10)
    .to_string(index=False)
)

item_meta shape: (15405, 31)

item_meta preview
 item_idx  movieId                 clean_title year_bucket                                                                         top_tags_text                                                                                                              movie_text
        1        1                   Toy Story   1991-2000                           pixar | animation | disney | tom hanks | computer animation     Toy Story. Adventure Animation Children Comedy Fantasy. pixar | animation | disney | tom hanks | computer animation
        2        2                     Jumanji   1991-2000                         robin williams | fantasy | time travel | animals | board game                      Jumanji. Adventure Children Fantasy. robin williams | fantasy | time travel | animals | board game
        3        3            Grumpier Old Men   1991-2000                        moldy | old | sequel | clv | comedinha de velhinhos engraã§ada             

In [14]:


# reduce genome features to a smaller vector
from sklearn.decomposition import TruncatedSVD

genome_small = genome_scores[genome_scores["movieId"].isin(movie_ids)].copy()

genome_wide = (
    genome_small.pivot(index="movieId", columns="tagId", values="relevance")
    .fillna(0)
    .astype("float32")
)

svd_dim = 64

svd = TruncatedSVD(n_components=svd_dim, random_state=42)
genome_reduced = svd.fit_transform(genome_wide)

genome_reduced = pd.DataFrame(
    genome_reduced,
    columns=[f"g{i}" for i in range(svd_dim)]
)
genome_reduced["movieId"] = genome_wide.index.values
genome_reduced["item_idx"] = genome_reduced["movieId"].map(item2idx)

genome_full = pd.DataFrame({"movieId": movie_ids})
genome_full["item_idx"] = genome_full["movieId"].map(item2idx)
genome_full = genome_full.merge(genome_reduced, on=["movieId", "item_idx"], how="left")
genome_full = genome_full.fillna(0).sort_values("item_idx").reset_index(drop=True)

print("movies with raw genome:", genome_wide.shape[0])
print("genome_full shape:", genome_full.shape)
print("svd explained variance sum:", round(svd.explained_variance_ratio_.sum(), 4))

print("\ngenome preview")
print(genome_full.head(5).to_string(index=False))

movies with raw genome: 10302
genome_full shape: (15405, 66)
svd explained variance sum: 0.7323

genome preview
 movieId  item_idx       g0       g1        g2        g3        g4        g5        g6       g7        g8        g9       g10       g11       g12       g13       g14       g15       g16       g17       g18       g19      g20       g21       g22       g23       g24       g25       g26       g27      g28       g29       g30       g31       g32       g33       g34       g35       g36       g37       g38       g39       g40       g41      g42       g43       g44       g45       g46       g47       g48       g49       g50       g51       g52       g53       g54       g55       g56       g57       g58      g59       g60       g61       g62       g63
       1         1 6.717240 1.186661 -2.051676  0.738828  0.900787  1.761749  1.217470 0.611673 -1.685390  0.833009  1.552775 -0.428605 -0.491475  0.214898 -1.183673 -0.719197  0.060525 -0.363902  0.404684  0.052617 0.250260 -0.693751 -

In [15]:


# save the processed objects
from pathlib import Path

OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

interactions.to_pickle(OUT_DIR / "interactions_35_kcore5.pkl")
splits.to_pickle(OUT_DIR / "splits_35_kcore5.pkl")
item_meta.to_pickle(OUT_DIR / "item_meta_35_kcore5.pkl")
genome_full.to_pickle(OUT_DIR / "genome_svd64_35_kcore5.pkl")

pd.DataFrame({
    "userId": list(user2idx.keys()),
    "user_idx": list(user2idx.values())
}).to_pickle(OUT_DIR / "user_map.pkl")

pd.DataFrame({
    "movieId": list(item2idx.keys()),
    "item_idx": list(item2idx.values())
}).to_pickle(OUT_DIR / "item_map.pkl")

print("saved files in:", OUT_DIR)
print(sorted([p.name for p in OUT_DIR.iterdir()]))

saved files in: ../data/processed
['genome_svd64_35_kcore5.pkl', 'interactions_35_kcore5.pkl', 'item_map.pkl', 'item_meta_35_kcore5.pkl', 'splits_35_kcore5.pkl', 'user_map.pkl']


_______
# 5 Model

Final modeling decision

We should use:

main task: next-item recommendation
base model: SASRec
interaction rule: rating >= 3.5
semantic branch: genres + year + raw tags + genome features when available
text semantic branch: later, from title + genres + top tags, but not in the very first training run
main evaluation: HR@10, NDCG@10, Recall@10
main split: chronological user split
main ablation ladder:
popularity baseline
MF/BPR baseline
SASRec-ID only
SASRec + structured semantics
SASRec + structured semantics + text semantics
Why choose rating >= 3.5, not >= 4.0 and not all ratings
Why not all ratings (>= 0.5)

Because that includes dislikes and weak signals as if they were positive interactions.

Second why:
transformer sequential recommenders for recommendation usually want a sequence of things the user likely wanted, not everything they touched. If we keep 0.5 and 1.0 ratings as positives, the model learns noisy preference sequences.

Why not >= 4.0

Because it gets too strict.

Second why:
your results show that with >= 4.0, the sequence lengths shrink more. That hurts transformer learning because the model has less history per user.
You already saw this clearly:

>= 3.5: median user sequence = 42
>= 4.0: median user sequence = 37

And for longer histories:

>= 3.5, users with seq >= 50: 40.50%
>= 4.0, users with seq >= 50: clearly lower
>= 3.5, users with seq >= 200: 7.62%
>= 4.0, even less

So >= 3.5 gives a better balance:

cleaner positive signal than all ratings
more data than >= 4.0

That is the sweet spot.

Why SASRec, not BERT4Rec first

Because your data now tells us you already have good sequence lengths for a causal sequential model.

At >= 3.5, median length is 42, 75th percentile is 100, and 90th percentile is 211. That is very usable for SASRec.

Second why:
SASRec is easier to train, easier to debug, and better for clean ablation work. For a research project, a stable baseline is more useful than starting with the harder model.

Why use semantics, and which semantics

Your semantic coverage is actually pretty good:

active movies: 26,744
active movies with raw tags: 19,011
active movies with genome: 10,370
active movies with both: 9,816

This means:

raw tags cover a lot
genome features are richer but only cover part of the catalog

So the right design is:

structured semantic branch

Use:

genres
year bucket
genome vector if available
fallback zeros or mask if genome missing
text semantic branch

Build movie text from:

clean title
genres
top raw tags
maybe top genome tags later

Why this combination:

it avoids depending only on genome, since genome misses many movies
it still uses strong semantic information where available
What your split results tell us

These are very good:

For >= 3.5:

train items: 22,768
val unseen item %: 0.04
test unseen item %: 0.04

That is nice because your validation and test almost never contain brand new items outside train. So this becomes a standard sequential recommendation setup, not a cold-start experiment.

Why this is good:

it makes the main results easier to compare with literature
it means poor performance later will mostly be model weakness, not split problems

In [16]:

# imports, config, load processed data
import math
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

PROC_DIR = Path("../data/processed")

interactions = pd.read_pickle(PROC_DIR / "interactions_35_kcore5.pkl")
splits = pd.read_pickle(PROC_DIR / "splits_35_kcore5.pkl")
item_meta = pd.read_pickle(PROC_DIR / "item_meta_35_kcore5.pkl")
genome_full = pd.read_pickle(PROC_DIR / "genome_svd64_35_kcore5.pkl")
item_map = pd.read_pickle(PROC_DIR / "item_map.pkl")
user_map = pd.read_pickle(PROC_DIR / "user_map.pkl")

num_users = len(user_map)
num_items = len(item_map)

MAX_LEN = 200
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 10

print("num_users:", num_users)
print("num_items:", num_items)
print("splits:", splits.shape)



device: cuda
num_users: 137474
num_items: 15405
splits: (137474, 7)


# Model 1 base SASRec

In [23]:

# small helpers and datasets
def pad_seq(seq, max_len=200):
    seq = seq[-max_len:]
    out = np.zeros(max_len, dtype=np.int64)
    out[-len(seq):] = seq
    return out

class TrainDataset(Dataset):
    def __init__(self, splits_df, max_len=200):
        self.max_len = max_len
        self.seqs = splits_df["train_seq"].tolist()

        keep = []
        for seq in self.seqs:
            if len(seq) >= 3:
                keep.append(seq)

        self.data = keep

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]

        tokens = seq[:-1]
        targets = seq[1:]

        tokens = pad_seq(tokens, self.max_len)
        targets = pad_seq(targets, self.max_len)

        return {
            "seq": torch.tensor(tokens, dtype=torch.long),
            "target_seq": torch.tensor(targets, dtype=torch.long),
        }

class EvalDataset(Dataset):
    def __init__(self, splits_df, mode="val", max_len=200):
        self.max_len = max_len
        self.mode = mode

        if mode == "val":
            self.seq_col = "val_seq"
            self.target_col = "val_target"
        else:
            self.seq_col = "test_seq"
            self.target_col = "test_target"

        self.seq = splits_df[self.seq_col].tolist()
        self.target = splits_df[self.target_col].tolist()

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return {
            "seq": torch.tensor(pad_seq(self.seq[idx], self.max_len), dtype=torch.long),
            "target": torch.tensor(self.target[idx], dtype=torch.long),
        }

train_ds = TrainDataset(splits, max_len=MAX_LEN)
val_ds = EvalDataset(splits, mode="val", max_len=MAX_LEN)
test_ds = EvalDataset(splits, mode="test", max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("train sequences:", len(train_ds))
print("val users:", len(val_ds))
print("test users:", len(test_ds))

train sequences: 137474
val users: 137474
test users: 137474


In [27]:


#SASRec base model
class SASRecBase(nn.Module):
    def __init__(self, num_items, max_len=200, d_model=128, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()

        self.num_items = num_items
        self.max_len = max_len
        self.d_model = d_model

        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len + 1, d_model, padding_idx=0)

        self.emb_dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.normal_(self.item_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)

    def _make_pos_ids(self, seq):
        mask = (seq > 0).long()
        pos_ids = torch.cumsum(mask, dim=1) * mask
        pos_ids = pos_ids.clamp(max=self.max_len)
        return pos_ids

    def _causal_mask(self, L, device):
        mask = torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)
        return mask

    def forward_sequence(self, seq):
        pos_ids = self._make_pos_ids(seq)
        x = self.item_emb(seq) + self.pos_emb(pos_ids)
        x = self.emb_norm(x)
        x = self.emb_dropout(x)

        pad_mask = (seq == 0)
        causal_mask = self._causal_mask(seq.size(1), seq.device)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        x = self.final_norm(x)
        return x

    def encode(self, seq):
        x = self.forward_sequence(seq)
        lengths = (seq > 0).sum(dim=1).clamp(min=1)
        last_idx = lengths - 1
        h = x[torch.arange(seq.size(0), device=seq.device), last_idx]
        return h

    def predict_all(self, seq):
        h = self.encode(seq)
        logits = h @ self.item_emb.weight.T
        logits[:, 0] = -1e9
        return logits

    def predict_all_positions(self, seq):
        x = self.forward_sequence(seq)
        logits = x @ self.item_emb.weight.T
        logits[:, :, 0] = -1e9
        return logits

model = SASRecBase(
    num_items=num_items,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(device)

print(model)

SASRecBase(
  (item_emb): Embedding(15406, 128, padding_idx=0)
  (pos_emb): Embedding(201, 128, padding_idx=0)
  (emb_dropout): Dropout(p=0.2, inplace=False)
  (emb_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)

In [28]:

# train and eval helpers
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target_seq = batch["target_seq"].to(device)

        optimizer.zero_grad()

        logits = model.predict_all_positions(seq)   # [B, L, num_items+1]

        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target_seq.reshape(-1),
            ignore_index=0
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        valid_tokens = (target_seq > 0).sum().item()
        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_topk(model, loader, device, k=10):
    model.eval()

    hits = 0.0
    ndcg = 0.0
    mrr = 0.0
    total = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target = batch["target"].to(device)

        logits = model.predict_all(seq)

        for i in range(seq.size(0)):
            seen = seq[i].unique()
            seen = seen[seen > 0]
            logits[i, seen] = -1e9

        topk = torch.topk(logits, k=k, dim=1).indices

        for i in range(seq.size(0)):
            tgt = target[i].item()
            pred = topk[i].tolist()

            if tgt in pred:
                rank = pred.index(tgt) + 1
                hits += 1.0
                ndcg += 1.0 / math.log2(rank + 1)
                mrr += 1.0 / rank

        total += seq.size(0)

    hr = hits / total
    ndcg = ndcg / total
    mrr = mrr / total

    return {
        f"HR@{k}": hr,
        f"NDCG@{k}": ndcg,
        f"MRR@{k}": mrr,
        f"Recall@{k}": hr,
    }

In [29]:

# train the base SASRec model
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
best_val_ndcg = -1
best_state = None

train_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate_topk(model, val_loader, device, k=10)

    epoch_time = time.perf_counter() - t0

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        **val_metrics
    }
    history.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg:
        best_val_ndcg = val_metrics["NDCG@10"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

total_train_sec = time.perf_counter() - train_start
print("\ntraining finished in", round(total_train_sec, 2), "seconds")

epoch 01 | loss 6.7041 | HR@10 0.0209 | NDCG@10 0.0098 | MRR@10 0.0064 | 191.6s
epoch 02 | loss 6.1218 | HR@10 0.0232 | NDCG@10 0.0110 | MRR@10 0.0073 | 191.8s
epoch 03 | loss 6.0189 | HR@10 0.0238 | NDCG@10 0.0115 | MRR@10 0.0078 | 192.8s
epoch 04 | loss 5.9657 | HR@10 0.0246 | NDCG@10 0.0119 | MRR@10 0.0081 | 191.8s
epoch 05 | loss 5.9331 | HR@10 0.0247 | NDCG@10 0.0120 | MRR@10 0.0081 | 191.9s
epoch 06 | loss 5.9106 | HR@10 0.0253 | NDCG@10 0.0123 | MRR@10 0.0084 | 191.7s
epoch 07 | loss 5.8942 | HR@10 0.0255 | NDCG@10 0.0125 | MRR@10 0.0086 | 191.4s
epoch 08 | loss 5.8816 | HR@10 0.0256 | NDCG@10 0.0124 | MRR@10 0.0085 | 191.9s
epoch 09 | loss 5.8716 | HR@10 0.0255 | NDCG@10 0.0124 | MRR@10 0.0084 | 191.9s
epoch 10 | loss 5.8635 | HR@10 0.0259 | NDCG@10 0.0126 | MRR@10 0.0086 | 191.7s

training finished in 1918.49 seconds


In [30]:

# load best epoch and evaluate on test
model.load_state_dict(best_state)

test_start = time.perf_counter()
test_metrics = evaluate_topk(model, test_loader, device, k=10)
test_sec = time.perf_counter() - test_start

print("best validation NDCG@10:", round(best_val_ndcg, 6))
print("test metrics:")
for k, v in test_metrics.items():
    print(k, ":", round(v, 6))
print("test eval seconds:", round(test_sec, 2))

history_df = pd.DataFrame(history)
history_df

best validation NDCG@10: 0.012623
test metrics:
HR@10 : 0.022884
NDCG@10 : 0.011223
MRR@10 : 0.007718
Recall@10 : 0.022884
test eval seconds: 48.77


,epoch,train_loss,epoch_sec,HR@10,NDCG@10,MRR@10,Recall@10
0,1,6.704104,191.577443,0.020877,0.009772,0.006448,0.020877
1,2,6.121785,191.775480,0.023161,0.010992,0.007348,0.023161
2,3,6.018896,192.835710,0.023830,0.011518,0.007827,0.023830
3,4,5.965745,191.760316,0.024557,0.011869,0.008066,0.024557
4,5,5.933090,191.864506,0.024732,0.011970,0.008143,0.024732
5,6,5.910595,191.739416,0.025336,0.012275,0.008353,0.025336
6,7,5.894190,191.420478,0.025547,0.012518,0.008597,0.025547
7,8,5.881551,191.854300,0.025561,0.012407,0.008459,0.025561
8,9,5.871609,191.864351,0.025503,0.012372,0.008427,0.025503
9,10,5.863467,191.733373,0.025889,0.012623,0.008637,0.025889


In [31]:


# save model and results
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = Path("../reports/results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(best_state, MODEL_DIR / "sasrec_base_id_only.pt")
history_df.to_csv(RESULT_DIR / "sasrec_base_history.csv", index=False)

pd.DataFrame([{
    "model": "sasrec_base_id_only",
    "num_users": num_users,
    "num_items": num_items,
    "max_len": MAX_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "epochs": EPOCHS,
    "train_total_sec": total_train_sec,
    "test_eval_sec": test_sec,
    **test_metrics
}]).to_csv(RESULT_DIR / "sasrec_base_test_metrics.csv", index=False)

print("saved model:", MODEL_DIR / "sasrec_base_id_only.pt")
print("saved history:", RESULT_DIR / "sasrec_base_history.csv")
print("saved test metrics:", RESULT_DIR / "sasrec_base_test_metrics.csv")

saved model: ../models/sasrec_base_id_only.pt
saved history: ../reports/results/sasrec_base_history.csv
saved test metrics: ../reports/results/sasrec_base_test_metrics.csv


________
# model 2 structured model

In [42]:

# S — build structured feature tensors
genre_cols = movies["genres"].str.get_dummies(sep="|").columns.tolist()
genre_cols = [c for c in genre_cols if c in item_meta.columns]

genome_cols = [c for c in genome_full.columns if c.startswith("g")]

year_vocab = [
    "unknown",
    "<=1950",
    "1951-1960",
    "1961-1970",
    "1971-1980",
    "1981-1990",
    "1991-2000",
    "2001-2010",
    "2011-2020",
]
year2idx = {y: i for i, y in enumerate(year_vocab)}

genre_mat = np.zeros((num_items + 1, len(genre_cols)), dtype=np.float32)
year_ids = np.zeros(num_items + 1, dtype=np.int64)
genome_mat = np.zeros((num_items + 1, len(genome_cols)), dtype=np.float32)
genome_flag = np.zeros(num_items + 1, dtype=np.int64)

meta_small = item_meta[["item_idx", "year_bucket"] + genre_cols].copy()
gen_small = genome_full[["item_idx"] + genome_cols].copy()

merged = meta_small.merge(gen_small, on="item_idx", how="left")

idx = merged["item_idx"].to_numpy()

genre_vals = merged[genre_cols].to_numpy(dtype=np.float32)
genre_mat[idx] = genre_vals

year_vals = merged["year_bucket"].map(year2idx).fillna(0).astype(int).to_numpy()
year_ids[idx] = year_vals

gen_vals = merged[genome_cols].fillna(0).to_numpy(dtype=np.float32)
genome_mat[idx] = gen_vals
genome_flag[idx] = (np.abs(gen_vals).sum(axis=1) > 0).astype(np.int64)

genre_tensor = torch.tensor(genre_mat, dtype=torch.float32)
year_tensor = torch.tensor(year_ids, dtype=torch.long)
genome_tensor = torch.tensor(genome_mat, dtype=torch.float32)
genome_flag_tensor = torch.tensor(genome_flag, dtype=torch.long)

print("genre tensor:", genre_tensor.shape)
print("year tensor:", year_tensor.shape)
print("genome tensor:", genome_tensor.shape)
print("genome flag tensor:", genome_flag_tensor.shape)
print("genre cols:", len(genre_cols))
print("genome cols:", len(genome_cols))
print("items with genome:", int(genome_flag.sum()))

genre tensor: torch.Size([15406, 20])
year tensor: torch.Size([15406])
genome tensor: torch.Size([15406, 64])
genome flag tensor: torch.Size([15406])
genre cols: 20
genome cols: 64
items with genome: 10302


In [43]:

# T — gated residual structured model
class SASRecStructuredGated(nn.Module):
    def __init__(
        self,
        num_items,
        genre_tensor,
        year_tensor,
        genome_tensor,
        genome_flag_tensor,
        max_len=200,
        d_model=128,
        n_heads=4,
        n_layers=2,
        dropout=0.2,
    ):
        super().__init__()

        self.num_items = num_items
        self.max_len = max_len
        self.d_model = d_model

        self.register_buffer("genre_features", genre_tensor)
        self.register_buffer("year_ids", year_tensor)
        self.register_buffer("genome_features", genome_tensor)
        self.register_buffer("genome_flags", genome_flag_tensor)

        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len + 1, d_model, padding_idx=0)

        self.genre_proj = nn.Linear(self.genre_features.shape[1], d_model)
        self.year_emb = nn.Embedding(int(self.year_ids.max().item()) + 1, d_model)
        self.genome_proj = nn.Linear(self.genome_features.shape[1], d_model)
        self.genome_flag_emb = nn.Embedding(2, d_model)

        self.semantic_norm = nn.LayerNorm(d_model)
        self.semantic_dropout = nn.Dropout(dropout)

        self.gate_net = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )

        self.emb_dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.normal_(self.item_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        nn.init.normal_(self.year_emb.weight, std=0.02)
        nn.init.normal_(self.genome_flag_emb.weight, std=0.02)

        nn.init.xavier_uniform_(self.genre_proj.weight)
        nn.init.zeros_(self.genre_proj.bias)

        nn.init.xavier_uniform_(self.genome_proj.weight)
        nn.init.zeros_(self.genome_proj.bias)

        nn.init.xavier_uniform_(self.gate_net[0].weight)
        nn.init.zeros_(self.gate_net[0].bias)
        nn.init.xavier_uniform_(self.gate_net[2].weight)
        nn.init.zeros_(self.gate_net[2].bias)

    def _make_pos_ids(self, seq):
        mask = (seq > 0).long()
        pos_ids = torch.cumsum(mask, dim=1) * mask
        pos_ids = pos_ids.clamp(max=self.max_len)
        return pos_ids

    def _causal_mask(self, L, device):
        return torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)

    def item_token_embed(self, seq):
        item_vec = self.item_emb(seq)

        semantic_vec = (
            self.genre_proj(self.genre_features[seq]) +
            self.year_emb(self.year_ids[seq]) +
            self.genome_proj(self.genome_features[seq]) +
            self.genome_flag_emb(self.genome_flags[seq])
        )

        semantic_vec = self.semantic_norm(semantic_vec)
        semantic_vec = self.semantic_dropout(semantic_vec)

        gate = self.gate_net(torch.cat([item_vec, semantic_vec], dim=-1))
        x = item_vec + gate * semantic_vec
        return x

    def forward_sequence(self, seq):
        pos_ids = self._make_pos_ids(seq)

        x = self.item_token_embed(seq) + self.pos_emb(pos_ids)
        x = self.emb_norm(x)
        x = self.emb_dropout(x)

        pad_mask = (seq == 0)
        causal_mask = self._causal_mask(seq.size(1), seq.device)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        x = self.final_norm(x)
        return x

    def encode(self, seq):
        x = self.forward_sequence(seq)
        lengths = (seq > 0).sum(dim=1).clamp(min=1)
        last_idx = lengths - 1
        h = x[torch.arange(seq.size(0), device=seq.device), last_idx]
        return h

    def predict_all(self, seq):
        h = self.encode(seq)
        logits = h @ self.item_emb.weight.T
        logits[:, 0] = -1e9
        return logits

    def predict_all_positions(self, seq):
        x = self.forward_sequence(seq)
        logits = x @ self.item_emb.weight.T
        logits[:, :, 0] = -1e9
        return logits

In [45]:

# U — train gated structured model with warm start
LR_STRUCT = 5e-4
EPOCHS_STRUCT = 6

structured_model = SASRecStructuredGated(
    num_items=num_items,
    genre_tensor=genre_tensor,
    year_tensor=year_tensor,
    genome_tensor=genome_tensor,
    genome_flag_tensor=genome_flag_tensor,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
).to(device)

def load_submodule_from_flat_state(submodule, state_dict, prefix):
    sub_state = {}
    for k, v in state_dict.items():
        if k.startswith(prefix):
            new_k = k[len(prefix):]
            sub_state[new_k] = v
    submodule.load_state_dict(sub_state)

# load base model weights safely
base_state = torch.load(
    MODEL_DIR / "sasrec_base_id_only.pt",
    map_location="cpu",
    weights_only=True
)

# warm start shared parts
structured_model.item_emb.load_state_dict({
    "weight": base_state["item_emb.weight"]
})

structured_model.pos_emb.load_state_dict({
    "weight": base_state["pos_emb.weight"]
})

structured_model.emb_norm.load_state_dict({
    "weight": base_state["emb_norm.weight"],
    "bias": base_state["emb_norm.bias"],
})

structured_model.final_norm.load_state_dict({
    "weight": base_state["final_norm.weight"],
    "bias": base_state["final_norm.bias"],
})

# load encoder layers from flat keys
for i in range(N_LAYERS):
    load_submodule_from_flat_state(
        structured_model.encoder.layers[i],
        base_state,
        prefix=f"encoder.layers.{i}."
    )

print(structured_model)

optimizer = torch.optim.Adam(structured_model.parameters(), lr=LR_STRUCT, weight_decay=WEIGHT_DECAY)

history_struct = []
best_val_ndcg_struct = -1
best_state_struct = None

train_start_struct = time.perf_counter()

for epoch in range(1, EPOCHS_STRUCT + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(structured_model, train_loader, optimizer, device)
    val_metrics = evaluate_topk(structured_model, val_loader, device, k=10)

    epoch_time = time.perf_counter() - t0

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        **val_metrics
    }
    history_struct.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg_struct:
        best_val_ndcg_struct = val_metrics["NDCG@10"]
        best_state_struct = {k: v.cpu().clone() for k, v in structured_model.state_dict().items()}

total_train_sec_struct = time.perf_counter() - train_start_struct
print("\ngated structured training finished in", round(total_train_sec_struct, 2), "seconds")

SASRecStructuredGated(
  (item_emb): Embedding(15406, 128, padding_idx=0)
  (pos_emb): Embedding(201, 128, padding_idx=0)
  (genre_proj): Linear(in_features=20, out_features=128, bias=True)
  (year_emb): Embedding(9, 128)
  (genome_proj): Linear(in_features=64, out_features=128, bias=True)
  (genome_flag_emb): Embedding(2, 128)
  (semantic_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (semantic_dropout): Dropout(p=0.2, inplace=False)
  (gate_net): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Sigmoid()
  )
  (emb_dropout): Dropout(p=0.2, inplace=False)
  (emb_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_featur

In [46]:

# V — evaluate gated structured model
structured_model.load_state_dict(best_state_struct)

test_start_struct = time.perf_counter()
test_metrics_struct = evaluate_topk(structured_model, test_loader, device, k=10)
test_sec_struct = time.perf_counter() - test_start_struct

print("best validation NDCG@10:", round(best_val_ndcg_struct, 6))
print("gated structured test metrics:")
for k, v in test_metrics_struct.items():
    print(k, ":", round(v, 6))
print("gated structured test eval seconds:", round(test_sec_struct, 2))

history_struct_df = pd.DataFrame(history_struct)
history_struct_df

best validation NDCG@10: 0.013073
gated structured test metrics:
HR@10 : 0.023503
NDCG@10 : 0.011681
MRR@10 : 0.008122
Recall@10 : 0.023503
gated structured test eval seconds: 49.96


,epoch,train_loss,epoch_sec,HR@10,NDCG@10,MRR@10,Recall@10
0,1,5.918146,197.688599,0.026318,0.012951,0.008932,0.026318
1,2,5.824056,197.241762,0.026507,0.013073,0.009039,0.026507
2,3,5.817753,197.449005,0.026441,0.012944,0.008886,0.026441
3,4,5.813547,197.338984,0.026427,0.012920,0.008860,0.026427
4,5,5.809793,197.257311,0.026398,0.012928,0.008883,0.026398
5,6,5.806655,197.397290,0.026412,0.012884,0.008824,0.026412


In [47]:


# W  save + compare
torch.save(best_state_struct, MODEL_DIR / "sasrec_structured_gated.pt")
history_struct_df.to_csv(RESULT_DIR / "sasrec_structured_gated_history.csv", index=False)

pd.DataFrame([{
    "model": "sasrec_structured_gated",
    "num_users": num_users,
    "num_items": num_items,
    "max_len": MAX_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "lr": LR_STRUCT,
    "weight_decay": WEIGHT_DECAY,
    "epochs": EPOCHS_STRUCT,
    "train_total_sec": total_train_sec_struct,
    "test_eval_sec": test_sec_struct,
    **test_metrics_struct
}]).to_csv(RESULT_DIR / "sasrec_structured_gated_test_metrics.csv", index=False)

compare_df = pd.DataFrame([
    {
        "model": "sasrec_base_id_only",
        "HR@10": test_metrics["HR@10"],
        "NDCG@10": test_metrics["NDCG@10"],
        "MRR@10": test_metrics["MRR@10"],
        "train_total_sec": total_train_sec,
        "test_eval_sec": test_sec,
    },
    {
        "model": "sasrec_structured_gated",
        "HR@10": test_metrics_struct["HR@10"],
        "NDCG@10": test_metrics_struct["NDCG@10"],
        "MRR@10": test_metrics_struct["MRR@10"],
        "train_total_sec": total_train_sec_struct,
        "test_eval_sec": test_sec_struct,
    }
])

print(compare_df)
compare_df.to_csv(RESULT_DIR / "compare_base_vs_structured_gated.csv", index=False)

print("saved model:", MODEL_DIR / "sasrec_structured_gated.pt")
print("saved history:", RESULT_DIR / "sasrec_structured_gated_history.csv")
print("saved test metrics:", RESULT_DIR / "sasrec_structured_gated_test_metrics.csv")
print("saved comparison:", RESULT_DIR / "compare_base_vs_structured_gated.csv")

                     model     HR@10   NDCG@10    MRR@10  train_total_sec  \
0      sasrec_base_id_only  0.022884  0.011223  0.007718      1918.489624   
1  sasrec_structured_gated  0.023503  0.011681  0.008122      1184.400158   

   test_eval_sec  
0      48.766527  
1      49.955825  
saved model: ../models/sasrec_structured_gated.pt
saved history: ../reports/results/sasrec_structured_gated_history.csv
saved test metrics: ../reports/results/sasrec_structured_gated_test_metrics.csv
saved comparison: ../reports/results/compare_base_vs_structured_gated.csv


_______
# Model 4 gated structured + text semantics
## Inspection

In [48]:

# X — clean and inspect movie text
import re
import unicodedata
import pandas as pd
import numpy as np

def clean_text_basic(x):
    x = "" if pd.isna(x) else str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("|", " ")
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x

item_text_df = item_meta[["item_idx", "movieId", "clean_title", "genres_text", "top_tags_text", "movie_text"]].copy()

item_text_df["movie_text_clean"] = (
    item_text_df["clean_title"].fillna("").map(clean_text_basic) + ". " +
    item_text_df["genres_text"].fillna("").map(clean_text_basic) + ". " +
    item_text_df["top_tags_text"].fillna("").map(clean_text_basic)
).str.strip()

item_text_df["char_len"] = item_text_df["movie_text_clean"].str.len()
item_text_df["word_len"] = item_text_df["movie_text_clean"].str.split().str.len()

print("text rows:", len(item_text_df))
print("empty text rows:", (item_text_df["movie_text_clean"].str.len() == 0).sum())

print("\nchar length quantiles")
print(item_text_df["char_len"].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nword length quantiles")
print(item_text_df["word_len"].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nsample cleaned text")
print(
    item_text_df[["movieId", "movie_text_clean"]]
    .head(10)
    .to_string(index=False)
)

text rows: 15405
empty text rows: 0

char length quantiles
0.25     61.0
0.50     84.0
0.75    101.0
0.90    117.0
0.95    130.0
0.99    158.0
Name: char_len, dtype: float64

word length quantiles
0.25     9.0
0.50    12.0
0.75    15.0
0.90    17.0
0.95    19.0
0.99    24.0
Name: word_len, dtype: float64

sample cleaned text
 movieId                                                                                                movie_text_clean
       1     toy story. adventure animation children comedy fantasy. pixar animation disney tom hanks computer animation
       2                      jumanji. adventure children fantasy. robin williams fantasy time travel animals board game
       3                        grumpier old men. comedy romance. moldy old sequel clv comedinha de velhinhos engraã§ada
       4                                     waiting to exhale. comedy drama romance. characters chick flick clv revenge
       5                   father of the bride part ii. comedy. stev

In [49]:

# Y — check if MiniLM text embeddings can load
TEXT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

try:
    from sentence_transformers import SentenceTransformer

    text_model = SentenceTransformer(TEXT_MODEL_NAME)
    sample_vec = text_model.encode(
        item_text_df["movie_text_clean"].head(3).tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    print("loaded text model:", TEXT_MODEL_NAME)
    print("embedding shape:", sample_vec.shape)

except Exception as e:
    text_model = None
    print("could not load sentence-transformer model")
    print(type(e).__name__, ":", e)

/workspace/movie-transformer-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded text model: sentence-transformers/all-MiniLM-L6-v2
embedding shape: (3, 384)


In [50]:

# Z — quick text quality preview on a few movies
sample_movie_ids = [1, 50, 260, 296, 318, 593, 2571]

preview = item_text_df[item_text_df["movieId"].isin(sample_movie_ids)][
    ["movieId", "movie_text_clean"]
].copy()

print(preview.to_string(index=False))

 movieId                                                                                                         movie_text_clean
       1              toy story. adventure animation children comedy fantasy. pixar animation disney tom hanks computer animation
      50              usual suspects, the. crime mystery thriller. twist ending kevin spacey suspense complicated organized crime
     260                star wars: episode iv - a new hope. action adventure sci-fi. sci-fi space harrison ford fantasy adventure
     296 pulp fiction. comedy crime drama thriller. quentin tarantino dark comedy nonlinear samuel l. jackson multiple storylines
     318                shawshank redemption, the. crime drama. morgan freeman prison twist ending stephen king thought-provoking
     593      silence of the lambs, the. crime horror thriller. serial killer psychology anthony hopkins cannibalism jodie foster
    2571                                matrix, the. action sci-fi thriller. sci-fi virtua


### Inspection result 

no empty rows
short and dense text
tags are meaningful
MiniLM loads fine

So the next model is:

gated structured + frozen MiniLM text embeddings

And we do it the safe way:

freeze the sentence model
precompute one 384-d text vector per movie
project that into the recommender space
add it with a separate gate
warm start from your best gated structured model

That is the clean next step.

Why this is the right next step
Why frozen text embeddings

Because the movie text is short and already informative.

Second why:
fine-tuning a language model inside the recommender would make the project much heavier and noisier. Frozen embeddings keep the experiment fair and easy to explain.

Why separate text gate

Because your earlier result already showed that naive fusion can hurt.

Second why:
if text helps, we want it to help only when useful, not overwrite the ID signal.

In [51]:

# AA — build MiniLM embeddings for all items
from sentence_transformers import SentenceTransformer
from pathlib import Path
import numpy as np
import pandas as pd
import torch

TEXT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TEXT_BATCH_SIZE = 256

text_model = SentenceTransformer(TEXT_MODEL_NAME)

texts = item_text_df.sort_values("item_idx")["movie_text_clean"].tolist()

text_emb = text_model.encode(
    texts,
    batch_size=TEXT_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("raw text embedding shape:", text_emb.shape)

text_emb_full = np.zeros((num_items + 1, text_emb.shape[1]), dtype=np.float32)
text_emb_full[1:] = text_emb.astype(np.float32)

text_emb_tensor = torch.tensor(text_emb_full, dtype=torch.float32)

print("full text tensor shape:", text_emb_tensor.shape)

Batches: 100%|██████████| 61/61 [00:02<00:00, 21.01it/s]

raw text embedding shape: (15405, 384)
full text tensor shape: torch.Size([15406, 384])


In [52]:

# AB — save text embeddings
TEXT_DIR = Path("../data/processed")
TEXT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(text_emb_tensor, TEXT_DIR / "text_minilm_tensor.pt")
print("saved:", TEXT_DIR / "text_minilm_tensor.pt")

saved: ../data/processed/text_minilm_tensor.pt


In [53]:

# AC — text + structured gated model
class SASRecStructuredTextGated(nn.Module):
    def __init__(
        self,
        num_items,
        genre_tensor,
        year_tensor,
        genome_tensor,
        genome_flag_tensor,
        text_tensor,
        max_len=200,
        d_model=128,
        n_heads=4,
        n_layers=2,
        dropout=0.2,
    ):
        super().__init__()

        self.num_items = num_items
        self.max_len = max_len
        self.d_model = d_model

        self.register_buffer("genre_features", genre_tensor)
        self.register_buffer("year_ids", year_tensor)
        self.register_buffer("genome_features", genome_tensor)
        self.register_buffer("genome_flags", genome_flag_tensor)
        self.register_buffer("text_features", text_tensor)

        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len + 1, d_model, padding_idx=0)

        self.genre_proj = nn.Linear(self.genre_features.shape[1], d_model)
        self.year_emb = nn.Embedding(int(self.year_ids.max().item()) + 1, d_model)
        self.genome_proj = nn.Linear(self.genome_features.shape[1], d_model)
        self.genome_flag_emb = nn.Embedding(2, d_model)

        self.text_proj = nn.Linear(self.text_features.shape[1], d_model)

        self.semantic_norm = nn.LayerNorm(d_model)
        self.semantic_dropout = nn.Dropout(dropout)

        self.text_norm = nn.LayerNorm(d_model)
        self.text_dropout = nn.Dropout(dropout)

        self.struct_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )

        self.text_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )

        self.emb_dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.normal_(self.item_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        nn.init.normal_(self.year_emb.weight, std=0.02)
        nn.init.normal_(self.genome_flag_emb.weight, std=0.02)

        nn.init.xavier_uniform_(self.genre_proj.weight)
        nn.init.zeros_(self.genre_proj.bias)

        nn.init.xavier_uniform_(self.genome_proj.weight)
        nn.init.zeros_(self.genome_proj.bias)

        nn.init.xavier_uniform_(self.text_proj.weight)
        nn.init.zeros_(self.text_proj.bias)

        nn.init.xavier_uniform_(self.struct_gate[0].weight)
        nn.init.zeros_(self.struct_gate[0].bias)
        nn.init.xavier_uniform_(self.struct_gate[2].weight)
        nn.init.zeros_(self.struct_gate[2].bias)

        nn.init.xavier_uniform_(self.text_gate[0].weight)
        nn.init.zeros_(self.text_gate[0].bias)
        nn.init.xavier_uniform_(self.text_gate[2].weight)
        nn.init.zeros_(self.text_gate[2].bias)

    def _make_pos_ids(self, seq):
        mask = (seq > 0).long()
        pos_ids = torch.cumsum(mask, dim=1) * mask
        pos_ids = pos_ids.clamp(max=self.max_len)
        return pos_ids

    def _causal_mask(self, L, device):
        return torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)

    def item_token_embed(self, seq):
        item_vec = self.item_emb(seq)

        struct_vec = (
            self.genre_proj(self.genre_features[seq]) +
            self.year_emb(self.year_ids[seq]) +
            self.genome_proj(self.genome_features[seq]) +
            self.genome_flag_emb(self.genome_flags[seq])
        )
        struct_vec = self.semantic_norm(struct_vec)
        struct_vec = self.semantic_dropout(struct_vec)

        text_vec = self.text_proj(self.text_features[seq])
        text_vec = self.text_norm(text_vec)
        text_vec = self.text_dropout(text_vec)

        g_struct = self.struct_gate(torch.cat([item_vec, struct_vec], dim=-1))
        g_text = self.text_gate(torch.cat([item_vec, text_vec], dim=-1))

        x = item_vec + g_struct * struct_vec + g_text * text_vec
        return x

    def forward_sequence(self, seq):
        pos_ids = self._make_pos_ids(seq)

        x = self.item_token_embed(seq) + self.pos_emb(pos_ids)
        x = self.emb_norm(x)
        x = self.emb_dropout(x)

        pad_mask = (seq == 0)
        causal_mask = self._causal_mask(seq.size(1), seq.device)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        x = self.final_norm(x)
        return x

    def encode(self, seq):
        x = self.forward_sequence(seq)
        lengths = (seq > 0).sum(dim=1).clamp(min=1)
        last_idx = lengths - 1
        h = x[torch.arange(seq.size(0), device=seq.device), last_idx]
        return h

    def predict_all(self, seq):
        h = self.encode(seq)
        logits = h @ self.item_emb.weight.T
        logits[:, 0] = -1e9
        return logits

    def predict_all_positions(self, seq):
        x = self.forward_sequence(seq)
        logits = x @ self.item_emb.weight.T
        logits[:, :, 0] = -1e9
        return logits

In [57]:

# AD — train text + structured gated model
LR_TEXT = 3e-4
EPOCHS_TEXT = 6

def load_submodule_from_flat_state(submodule, state_dict, prefix):
    sub_state = {}
    for k, v in state_dict.items():
        if k.startswith(prefix):
            sub_state[k[len(prefix):]] = v
    submodule.load_state_dict(sub_state)

text_tensor = torch.load("../data/processed/text_minilm_tensor.pt", map_location="cpu")

text_model_rec = SASRecStructuredTextGated(
    num_items=num_items,
    genre_tensor=genre_tensor,
    year_tensor=year_tensor,
    genome_tensor=genome_tensor,
    genome_flag_tensor=genome_flag_tensor,
    text_tensor=text_tensor,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
).to(device)

gated_state = torch.load(
    MODEL_DIR / "sasrec_structured_gated.pt",
    map_location="cpu",
    weights_only=True
)

# shared embeddings + norms
text_model_rec.item_emb.load_state_dict({
    "weight": gated_state["item_emb.weight"]
})
text_model_rec.pos_emb.load_state_dict({
    "weight": gated_state["pos_emb.weight"]
})
text_model_rec.emb_norm.load_state_dict({
    "weight": gated_state["emb_norm.weight"],
    "bias": gated_state["emb_norm.bias"],
})
text_model_rec.final_norm.load_state_dict({
    "weight": gated_state["final_norm.weight"],
    "bias": gated_state["final_norm.bias"],
})

# encoder blocks
for i in range(N_LAYERS):
    load_submodule_from_flat_state(
        text_model_rec.encoder.layers[i],
        gated_state,
        prefix=f"encoder.layers.{i}."
    )

# structured branch
text_model_rec.genre_proj.load_state_dict({
    "weight": gated_state["genre_proj.weight"],
    "bias": gated_state["genre_proj.bias"],
})
text_model_rec.year_emb.load_state_dict({
    "weight": gated_state["year_emb.weight"],
})
text_model_rec.genome_proj.load_state_dict({
    "weight": gated_state["genome_proj.weight"],
    "bias": gated_state["genome_proj.bias"],
})
text_model_rec.genome_flag_emb.load_state_dict({
    "weight": gated_state["genome_flag_emb.weight"],
})
text_model_rec.semantic_norm.load_state_dict({
    "weight": gated_state["semantic_norm.weight"],
    "bias": gated_state["semantic_norm.bias"],
})

# old gated model used gate_net, new text model uses struct_gate
text_model_rec.struct_gate.load_state_dict({
    "0.weight": gated_state["gate_net.0.weight"],
    "0.bias": gated_state["gate_net.0.bias"],
    "2.weight": gated_state["gate_net.2.weight"],
    "2.bias": gated_state["gate_net.2.bias"],
})

# text branch is new, so it stays randomly initialized:
# - text_proj
# - text_norm
# - text_gate

print(text_model_rec)

optimizer = torch.optim.Adam(text_model_rec.parameters(), lr=LR_TEXT, weight_decay=WEIGHT_DECAY)

history_text = []
best_val_ndcg_text = -1
best_state_text = None

train_start_text = time.perf_counter()

for epoch in range(1, EPOCHS_TEXT + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(text_model_rec, train_loader, optimizer, device)
    val_metrics = evaluate_topk(text_model_rec, val_loader, device, k=10)

    epoch_time = time.perf_counter() - t0

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        **val_metrics
    }
    history_text.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg_text:
        best_val_ndcg_text = val_metrics["NDCG@10"]
        best_state_text = {k: v.cpu().clone() for k, v in text_model_rec.state_dict().items()}

total_train_sec_text = time.perf_counter() - train_start_text
print("\ntext+structured training finished in", round(total_train_sec_text, 2), "seconds")

/tmp/ipykernel_2311/2148865399.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  text_tensor = torch.load("../data/processed/text_minilm_tensor.pt", map_location="cpu")


SASRecStructuredTextGated(
  (item_emb): Embedding(15406, 128, padding_idx=0)
  (pos_emb): Embedding(201, 128, padding_idx=0)
  (genre_proj): Linear(in_features=20, out_features=128, bias=True)
  (year_emb): Embedding(9, 128)
  (genome_proj): Linear(in_features=64, out_features=128, bias=True)
  (genome_flag_emb): Embedding(2, 128)
  (text_proj): Linear(in_features=384, out_features=128, bias=True)
  (semantic_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (semantic_dropout): Dropout(p=0.2, inplace=False)
  (text_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (text_dropout): Dropout(p=0.2, inplace=False)
  (struct_gate): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Sigmoid()
  )
  (text_gate): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_feat

In [58]:

# AE — evaluate text + structured gated model
text_model_rec.load_state_dict(best_state_text)

test_start_text = time.perf_counter()
test_metrics_text = evaluate_topk(text_model_rec, test_loader, device, k=10)
test_sec_text = time.perf_counter() - test_start_text

print("best validation NDCG@10:", round(best_val_ndcg_text, 6))
print("text+structured test metrics:")
for k, v in test_metrics_text.items():
    print(k, ":", round(v, 6))
print("text+structured test eval seconds:", round(test_sec_text, 2))

history_text_df = pd.DataFrame(history_text)
history_text_df

best validation NDCG@10: 0.013185
text+structured test metrics:
HR@10 : 0.023524
NDCG@10 : 0.011794
MRR@10 : 0.008261
Recall@10 : 0.023524
text+structured test eval seconds: 52.2


,epoch,train_loss,epoch_sec,HR@10,NDCG@10,MRR@10,Recall@10
0,1,5.925337,203.281206,0.026347,0.012983,0.008954,0.026347
1,2,5.805949,203.328240,0.026587,0.013086,0.009030,0.026587
2,3,5.800860,203.437866,0.026529,0.013119,0.009086,0.026529
3,4,5.796763,203.804781,0.026623,0.013185,0.009136,0.026623
4,5,5.793913,204.017974,0.026514,0.013102,0.009068,0.026514
5,6,5.791395,204.071625,0.026718,0.013128,0.009036,0.026718


In [60]:


# AF — save + compare all models
torch.save(best_state_text, MODEL_DIR / "sasrec_structured_text_gated.pt")
history_text_df.to_csv(RESULT_DIR / "sasrec_structured_text_gated_history.csv", index=False)

pd.DataFrame([{
    "model": "sasrec_structured_text_gated",
    "num_users": num_users,
    "num_items": num_items,
    "max_len": MAX_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "lr": LR_TEXT,
    "weight_decay": WEIGHT_DECAY,
    "epochs": EPOCHS_TEXT,
    "train_total_sec": total_train_sec_text,
    "test_eval_sec": test_sec_text,
    **test_metrics_text
}]).to_csv(RESULT_DIR / "sasrec_structured_text_gated_test_metrics.csv", index=False)

# put the real naive structured numbers manually from the earlier run
naive_struct_metrics = {
    "HR@10": 0.022339,
    "NDCG@10": 0.010935,
    "MRR@10": 0.007522,
    "train_total_sec": 1966.953740,
    "test_eval_sec": 49.775926,
}

compare_all_df = pd.DataFrame([
    {
        "model": "sasrec_base_id_only",
        "HR@10": test_metrics["HR@10"],
        "NDCG@10": test_metrics["NDCG@10"],
        "MRR@10": test_metrics["MRR@10"],
        "train_total_sec": total_train_sec,
        "test_eval_sec": test_sec,
    },
    {
        "model": "sasrec_structured_naive",
        "HR@10": naive_struct_metrics["HR@10"],
        "NDCG@10": naive_struct_metrics["NDCG@10"],
        "MRR@10": naive_struct_metrics["MRR@10"],
        "train_total_sec": naive_struct_metrics["train_total_sec"],
        "test_eval_sec": naive_struct_metrics["test_eval_sec"],
    },
    {
        "model": "sasrec_structured_gated",
        "HR@10": test_metrics_struct["HR@10"],
        "NDCG@10": test_metrics_struct["NDCG@10"],
        "MRR@10": test_metrics_struct["MRR@10"],
        "train_total_sec": total_train_sec_struct,
        "test_eval_sec": test_sec_struct,
    },
    {
        "model": "sasrec_structured_text_gated",
        "HR@10": test_metrics_text["HR@10"],
        "NDCG@10": test_metrics_text["NDCG@10"],
        "MRR@10": test_metrics_text["MRR@10"],
        "train_total_sec": total_train_sec_text,
        "test_eval_sec": test_sec_text,
    }
])

print(compare_all_df)
compare_all_df.to_csv(RESULT_DIR / "compare_all_models_clean.csv", index=False)

print("saved model:", MODEL_DIR / "sasrec_structured_text_gated.pt")
print("saved history:", RESULT_DIR / "sasrec_structured_text_gated_history.csv")
print("saved test metrics:", RESULT_DIR / "sasrec_structured_text_gated_test_metrics.csv")
print("saved comparison:", RESULT_DIR / "compare_all_models_clean.csv")

                          model     HR@10   NDCG@10    MRR@10  \
0           sasrec_base_id_only  0.022884  0.011223  0.007718   
1       sasrec_structured_naive  0.022339  0.010935  0.007522   
2       sasrec_structured_gated  0.023503  0.011681  0.008122   
3  sasrec_structured_text_gated  0.023524  0.011794  0.008261   

   train_total_sec  test_eval_sec  
0      1918.489624      48.766527  
1      1966.953740      49.775926  
2      1184.400158      49.955825  
3      1222.056188      52.201269  
saved model: ../models/sasrec_structured_text_gated.pt
saved history: ../reports/results/sasrec_structured_text_gated_history.csv
saved test metrics: ../reports/results/sasrec_structured_text_gated_test_metrics.csv
saved comparison: ../reports/results/compare_all_models_clean.csv


### Comment
Also, the newest run is the best model so far. The attached output shows:

base SASRec: HR@10 0.022884, NDCG@10 0.011223, MRR@10 0.007718
gated structured: HR@10 0.023503, NDCG@10 0.011681, MRR@10 0.008122
structured + text gated: HR@10 0.023524, NDCG@10 0.011794, MRR@10 0.008261

So the ranking is now clear:

structured + text gated = best
gated structured = second
base SASRec = third
naive structured fusion = worse than gated and only useful as a negative ablation result
Important correction in the current comparison table

The current compare_all_df has a labeling mistake:

row "sasrec_structured" is actually showing the gated structured numbers
row "sasrec_structured_gated" shows the same numbers again

So the table is useful for the best model, but the middle rows need cleanup before using them in the report.

What the results mean

This is a nice story for the report:

Base SASRec works
Naive feature fusion did not help
Gated structured fusion helped
Adding frozen text embeddings helped a bit more

That is a strong research narrative because it shows the main point clearly:

extra semantic information is useful, but only when fusion is controlled

Why the text gain is small but still valuable

The jump from gated structured to structured+text gated is not huge, but it is real:

NDCG@10: 0.011681 -> 0.011794
MRR@10: 0.008122 -> 0.008261

Why small:

the item-ID signal is already strong after filtering
the structured branch already captures part of the movie meaning, so text adds refinement, not a full new signal

That is still a good result, because small gains are normal once the baseline is already decent.

In [ ]:

#